# 01 — Baseline (clean)

Wrapper над исходным ноутбуком (эксперимент `01_baseline_clean`). Исходник не изменяется.

> **Замечание о воспроизводимости.** Эти wrapper-ноутбуки запускают исходные `fin_*.ipynb` *как есть*.
> Для запуска нужны: (1) сами исходные `.ipynb`, (2) папка-репозиторий `tpe/` в Google Drive
> (`drive/MyDrive/content/tpe`), (3) `ConfigSpace==1.2.0`, `parzen-estimator`, `optuna`, `scipy`.
> Пакет `tpe` **не входит** в git-репозиторий курсовой — он лежит только в Drive. Поэтому без Drive код не воспроизводится.
> Параметры (SEEDS=range(30), MAX_EVALS=100, BASE_KWARGS) **зашиты внутри исходников**: wrapper их не переопределяет,
> чтобы не нарушать правило 'не менять исходный код'. Унификация здесь — это единый запуск, единое хранение и единый сбор метрик.

## Разбор эксперимента
**Цель:** показать, что repo-TPE без модификаций ведёт себя осмысленно (лучше random, сопоставимо с Optuna).

**Способ изменения ТПЕ:** отсутствует — это контрольная точка.

**Варианты:** `random`, `no_w` (repo-TPE), `optuna`.

**Ключевой параметр-особенность:** `n_init = 25` (в остальных экспериментах 10) и **мягкие пороги**
(Sphere 1e-2, Rosenbrock 1e-1, Rastrigin 2.0, Ackley 0.5).

**Реальные результаты из исходника (success_rate %, 30 seeds):**
Sphere: no_w 60.0, optuna 43.3, random 3.3; Rosenbrock: no_w 53.3, optuna 30.0, random 23.3;
Rastrigin: no_w 56.7, optuna 36.7, random 3.3; Ackley: no_w 23.3, optuna 36.7, random 3.3.

**Вывод, который МОЖНО делать:** repo-TPE стабильно бьёт random; на 3 из 4 функций обгоняет и Optuna
(при n_init=25 и мягких порогах).

**Чего делать НЕЛЬЗЯ:** сравнивать эти числа напрямую с 03/04/05 — там другой n_init и более жёсткие пороги.

In [ ]:
# === Единая настройка путей и окружения (общая для всех wrapper-ноутбуков) ===
# ВАЖНО: этот блок НЕ трогает исходные ноутбуки. Он только готовит окружение,
# монтирует Drive и определяет, где лежат (а) исходные .ipynb и (б) папка-репозиторий tpe.
import os, sys, json, shutil, subprocess, time, glob
from pathlib import Path

# 1) Монтируем Google Drive (в Colab). Локально просто пропустится.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', repr(e))

# 2) Где лежат ИСХОДНЫЕ ноутбуки (5 файлов fin_*.ipynb).
#    Поменяйте при необходимости на свой путь в Drive.
ORIGINALS_DIR = Path(os.environ.get('ORIGINALS_DIR', 'drive/MyDrive/content/k5_originals'))
if not ORIGINALS_DIR.exists():
    for cand in ['/content/drive/MyDrive/content/k5_originals',
                 'drive/MyDrive/content', '/content/drive/MyDrive/content', '.']:
        if list(Path(cand).glob('fin_*.ipynb')):
            ORIGINALS_DIR = Path(cand); break

# 3) Корень анализа (эта папка coursework_analysis).
ANALYSIS_ROOT = Path(os.environ.get('ANALYSIS_ROOT', '.')).resolve()
for cand in [Path('.'), Path('coursework_analysis'), Path('/content/coursework_analysis')]:
    if (cand / 'configs' / 'common_config.json').exists():
        ANALYSIS_ROOT = cand.resolve(); break

RAW_DIR   = ANALYSIS_ROOT / 'results' / 'raw'
PROC_DIR  = ANALYSIS_ROOT / 'results' / 'processed'
TABLES_DIR= ANALYSIS_ROOT / 'tables'
FIG_DIR   = ANALYSIS_ROOT / 'figures'
for d in (RAW_DIR, PROC_DIR, TABLES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

COMMON = json.load(open(ANALYSIS_ROOT / 'configs' / 'common_config.json'))
EXPS   = json.load(open(ANALYSIS_ROOT / 'configs' / 'experiments_config.json'))
EXP_BY_ID = {e['id']: e for e in EXPS['experiments']}

print('ORIGINALS_DIR =', ORIGINALS_DIR)
print('ANALYSIS_ROOT =', ANALYSIS_ROOT)
print('originals found:', sorted(p.name for p in ORIGINALS_DIR.glob('fin_*.ipynb')))

In [ ]:
# === Единый запуск исходного ноутбука БЕЗ его изменения + сбор результатов ===
# Стратегия: исходные ноутбуки самодостаточны (ставят зависимости, монтируют Drive,
# импортируют repo-tpe, считают и СОХРАНЯЮТ свои CSV в ./results/*.csv).
# Мы выполняем их 'как есть' через nbconvert --execute (копия в /tmp, оригинал не трогаем),
# затем переносим произведённые CSV в coursework_analysis/results/raw/ под унифицированными именами.

def run_original_and_harvest(exp_id, fix_seed=0, timeout=7200):
    exp = EXP_BY_ID[exp_id]
    src = ORIGINALS_DIR / exp['source_notebook']
    assert src.exists(), f'Не найден исходный ноутбук: {src}'

    work = Path('/content/_run') if Path('/content').exists() else Path('/tmp/_run')
    work.mkdir(parents=True, exist_ok=True)
    tmp_nb = work / src.name
    shutil.copy(src, tmp_nb)

    # Лог метаданных запуска (seed/versions). Сам seed зашит в исходниках (SEEDS=range(30)).
    meta = {'exp_id': exp_id, 'source': str(src), 'fix_seed_requested': fix_seed,
            'started': time.strftime('%Y-%m-%d %H:%M:%S')}
    try:
        import numpy, optuna, ConfigSpace
        meta['versions'] = {'numpy': numpy.__version__,
                            'optuna': getattr(optuna,'__version__','?'),
                            'ConfigSpace': getattr(ConfigSpace,'__version__','?')}
    except Exception as e:
        meta['versions_error'] = repr(e)

    print(f'Executing {src.name} ... (может занять минуты)')
    cmd = ['jupyter','nbconvert','--to','notebook','--execute',
           '--ExecutePreprocessor.timeout=%d'%timeout, str(tmp_nb),
           '--output', tmp_nb.stem + '_executed']
    res = subprocess.run(cmd, cwd=str(work), capture_output=True, text=True)
    meta['nbconvert_returncode'] = res.returncode
    if res.returncode != 0:
        print('STDERR tail:\n', res.stderr[-2000:])
    # Исходники сохраняют CSV в <cwd>/results/. cwd = work.
    produced = list((work / 'results').glob('*.csv')) if (work/'results').exists() else []
    harvested = []
    for produced_name, target_name in zip(exp['produces_csv'], exp['harvest_as']):
        p = work / produced_name
        if p.exists():
            shutil.copy(p, RAW_DIR / target_name)
            harvested.append(target_name)
    meta['harvested'] = harvested
    meta['finished'] = time.strftime('%Y-%m-%d %H:%M:%S')
    json.dump(meta, open(RAW_DIR / f'{exp_id}_runmeta.json','w'), ensure_ascii=False, indent=2)
    print('Harvested ->', harvested)
    return meta

In [ ]:
EXP_ID = '01_baseline_clean'
exp = EXP_BY_ID[EXP_ID]
print('Source notebook:', exp['source_notebook'])
print('TPE modification:', exp['tpe_modification'])
print('Variants:', exp['variants'])

### Запуск (раскомментируйте, когда Drive и пакет `tpe` доступны)

In [ ]:
# meta = run_original_and_harvest(EXP_ID)
# meta

### Быстрый просмотр собранной сводки

In [ ]:
import pandas as pd
for name in exp['harvest_as']:
    p = RAW_DIR / name
    if p.exists() and p.suffix=='.csv':
        print('\n===', name, '===')
        display(pd.read_csv(p).head(20))
    else:
        print('нет файла (ещё не запускали):', name)